# DEEP-GUARD Phase 2 — OpenFake Generalization Experiment

This notebook trains a separate candidate model using a broader generator distribution. It preserves the existing Tiny-GenImage checkpoint and scores. The main test is OpenFake's source-defined OOD split; a separate `reddit/test` split measures in-the-wild transfer. Neither test split is used for threshold calibration or model selection.

**License:** review the [OpenFake dataset card](https://huggingface.co/datasets/ComplexDataLab/OpenFake), including generator-specific terms. This notebook does not publish the resulting checkpoint to GitHub; it packages the candidate and metrics for local academic evaluation. Do not redistribute until terms are clear.

Sample plan: 4,000/class train, 500/class validation, 800/class OOD test, 300/class in-the-wild test. GPU required.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json, torch

REPO = Path('/content/DEEP-GUARD')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ABHILATTHE6/DEEP-GUARD.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
os.chdir(REPO)
print('Repository:', REPO)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Enable a GPU in Colab: Runtime > Change runtime type > T4 GPU, then rerun this cell.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[ml,dev]'], check=True)


## 1. Restore the existing checkpoint as the comparison baseline

This downloads the current public checkpoint into a separate path. It will not be overwritten.

In [ ]:
DATA_ROOT = Path('/content/deepguard-openfake')
OUT = Path('/content/deepguard-openfake-artifacts')
OUT.mkdir(parents=True, exist_ok=True)
BASELINE_CKPT = OUT / 'tinygenimage-baseline-best.pt'
CANDIDATE_LAST = OUT / 'openfake-last.pt'
CANDIDATE_BEST = OUT / 'openfake-best.pt'

subprocess.run([
    sys.executable, 'scripts/github_release_checkpoint.py', 'download',
    '--repo', 'ABHILATTHE6/DEEP-GUARD', '--tag', 'training-checkpoints',
    '--asset-name', 'image-training-best.pt', '--output', str(BASELINE_CKPT)
], check=True)
print('Baseline:', BASELINE_CKPT, 'bytes:', BASELINE_CKPT.stat().st_size)


## 2. Prepare a diverse train/validation/OOD/wild sample

The script streams from OpenFake's `core` and `reddit` configs, saves only the requested samples, and writes the source split and source-model summary to the manifest. Leave this cell running until it finishes.

In [ ]:
prep_cmd = [
    sys.executable, 'scripts/prepare_openfake_hf.py',
    '--output', str(DATA_ROOT),
    '--train-per-class', '4000',
    '--val-per-class', '500',
    '--ood-test-per-class', '800',
    '--wild-test-per-class', '300',
    '--seed', '4201',
    '--buffer-size', '20000',
]
subprocess.run(prep_cmd, check=True)
manifest = json.loads((DATA_ROOT / 'dataset_manifest.json').read_text(encoding='utf-8'))
print(json.dumps(manifest['sample_counts'], indent=2))
print('Manifest:', DATA_ROOT / 'dataset_manifest.json')


## 3. Evaluate the current baseline on the new held-out test sets

This measures how the published model transfers to OpenFake. The evaluation script calibrates the threshold on `core/validation` only, then evaluates on OOD and in-the-wild splits.

In [ ]:
BASELINE_OOD = OUT / 'baseline_openfake_ood.json'
BASELINE_WILD = OUT / 'baseline_openfake_wild.json'

def evaluate(checkpoint, test_dir, output_file):
    cmd = [
        sys.executable, 'scripts/evaluate_image.py', '--checkpoint', str(checkpoint),
        '--val', str(DATA_ROOT / 'val'), '--test', str(DATA_ROOT / test_dir),
        '--output', str(output_file), '--batch-size', '64', '--target-real-fpr', '0.03',
    ]
    subprocess.run(cmd, check=True)

evaluate(BASELINE_CKPT, 'test_ood', BASELINE_OOD)
evaluate(BASELINE_CKPT, 'test_wild', BASELINE_WILD)


## 4. Train a separate candidate checkpoint

This trains on 8,000 OpenFake training images for five epochs. Candidate files use distinct names and remain in the runtime; this cell does not upload/overwrite public checkpoint assets. If the runtime resets, rerun the preparation/training cells or download the result ZIP after finishing.

In [ ]:
train_cmd = [
    sys.executable, 'scripts/train_image.py',
    '--train', str(DATA_ROOT / 'train'), '--val', str(DATA_ROOT / 'val'),
    '--epochs', '5', '--batch-size', '64', '--lr', '1e-4',
    '--weight-decay', '1e-4', '--num-workers', '2', '--seed', '4201',
    '--target-real-fpr', '0.03', '--output', str(CANDIDATE_LAST),
    '--best-output', str(CANDIDATE_BEST),
]
subprocess.run(train_cmd, check=True)
assert CANDIDATE_BEST.is_file(), 'Training did not produce the best candidate checkpoint.'
print('Candidate checkpoint:', CANDIDATE_BEST, 'bytes:', CANDIDATE_BEST.stat().st_size)


## 5. Evaluate the candidate on the exact same held-out test images

Each model's threshold is calibrated on the same `core/validation` subset to target the same validation real-image false-positive rate. Neither test split affects threshold selection.

In [ ]:
CANDIDATE_OOD = OUT / 'candidate_openfake_ood.json'
CANDIDATE_WILD = OUT / 'candidate_openfake_wild.json'
evaluate(CANDIDATE_BEST, 'test_ood', CANDIDATE_OOD)
evaluate(CANDIDATE_BEST, 'test_wild', CANDIDATE_WILD)

import pandas as pd
rows = []
for model_name, reports in [
    ('Current Tiny-GenImage baseline', {'OOD test': BASELINE_OOD, 'In-the-wild test': BASELINE_WILD}),
    ('OpenFake candidate', {'OOD test': CANDIDATE_OOD, 'In-the-wild test': CANDIDATE_WILD}),
]:
    for test_name, report_path in reports.items():
        report = json.loads(report_path.read_text(encoding='utf-8'))
        m = report['test']
        rows.append({
            'Model': model_name, 'Test': test_name, 'n': m['count'],
            'Argmax accuracy': m['argmax_accuracy'],
            'Calibrated accuracy': m['calibrated_accuracy'],
            'AI precision': m['ai_precision'], 'AI recall': m['ai_recall'],
            'AI F1': m['ai_f1'], 'ROC-AUC': m['roc_auc_ai_positive'],
            'Real false-positive rate': m['real_false_positive_rate'],
            'Validation threshold': report['real_probability_threshold'],
        })
display(pd.DataFrame(rows).style.format({c: '{:.3f}' for c in ['Argmax accuracy','Calibrated accuracy','AI precision','AI recall','AI F1','ROC-AUC','Real false-positive rate','Validation threshold']}))
print('Promote the candidate only if results on the same held-out splits justify it.')


## 6. Package the experiment for download

The ZIP contains the separate candidate checkpoint, JSON evaluation reports, and data manifest. It excludes raw images and does not write a new checkpoint to the public GitHub release. Review dataset terms before sharing or redistributing the candidate model.

In [ ]:
import hashlib, zipfile
from google.colab import files

summary = {
    'experiment': 'DEEP-GUARD OpenFake candidate',
    'baseline_checkpoint_sha256': hashlib.sha256(BASELINE_CKPT.read_bytes()).hexdigest(),
    'candidate_checkpoint_sha256': hashlib.sha256(CANDIDATE_BEST.read_bytes()).hexdigest(),
    'candidate_checkpoint': CANDIDATE_BEST.name,
    'dataset': 'ComplexDataLab/OpenFake',
    'license_note': 'Review the dataset card and generator-specific terms; do not redistribute until cleared.',
}
(OUT / 'experiment_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
bundle = Path('/content/DEEP_GUARD_OpenFake_Experiment_Results.zip')
include = [CANDIDATE_BEST, DATA_ROOT / 'dataset_manifest.json', BASELINE_OOD, BASELINE_WILD, CANDIDATE_OOD, CANDIDATE_WILD, OUT / 'experiment_summary.json']
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in include:
        zf.write(path, arcname=path.name)
print('Results bundle:', bundle, 'bytes:', bundle.stat().st_size)
files.download(str(bundle))
